In [1]:
from datasets import load_dataset
dataset = load_dataset("UBC-SLIME/colx_531_group_project")

/Users/jennifer/miniforge3/envs/tcob/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoModelForCausalLM

lm1 = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")

lm2 = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-360m")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 18544.44it/s]


In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'is_member'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['id', 'text', 'is_member'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['id', 'text', 'is_member'],
        num_rows: 15000
    })
})

In [4]:
import pandas as pd
import altair as alt
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import math

# Configure Altair
alt.data_transformers.enable('json')

DataTransformerRegistry.enable('json')

# Membership Inference Attack - EDA

**Goal:** Identify "memorization signatures" that distinguish texts used to fine-tune the models (members) from those that weren't (non-members).

We'll focus on patterns that could indicate memorization rather than generic text statistics.

In [5]:
# Quick Dataset Overview
print(f"Training set: {len(dataset['train'])} samples")
print(f"Class balance: {dataset['train'].to_pandas()['is_member'].value_counts().to_dict()}")

# Convert to pandas for analysis
train_df = dataset['train'].to_pandas()
print(f"\\nSample record:")
print(f"Text preview: {train_df.iloc[0]['text'][:200]}...")
print(f"Is member: {train_df.iloc[0]['is_member']}")

Training set: 50000 samples
Class balance: {0: 25000, 1: 25000}
\nSample record:
Text preview: Discharge Summary:
Patient Name: N/A
Patient ID: N/A
Gender: Female
Age: 43
Admit Date: 29th January 2018
Discharge Date: N/A

Chief Complaint:
Infertility evaluation and management.

Hospital Course ...
Is member: 0


In [6]:
# Quick Class Distribution Check
class_counts = train_df['is_member'].value_counts()
print(f"Class distribution: {class_counts.to_dict()}")
print(f"Dataset is {'balanced' if abs(class_counts[0] - class_counts[1]) < 1000 else 'imbalanced'}")

# Split data for subsequent analysis
member_texts = train_df[train_df['is_member'] == 1]
non_member_texts = train_df[train_df['is_member'] == 0]
print(f"\\nMember texts: {len(member_texts)}, Non-member texts: {len(non_member_texts)}")

Class distribution: {0: 25000, 1: 25000}
Dataset is balanced
\nMember texts: 25000, Non-member texts: 25000


# EDA for Membership Inference

The goal is to find "memorization signatures" that distinguish training data from non-training data.

In [7]:
#  Text Uniqueness and Rarity Analysis

def analyze_text_uniqueness(df):
    """Analyze text uniqueness and rarity patterns"""
    
    # Calculate text-level metrics
    df = df.copy()
    df['text_lower'] = df['text'].str.lower()
    
    # Exact duplicates
    df['is_duplicate'] = df.duplicated(subset=['text_lower'], keep=False)
    
    # Near duplicates 
    def get_char_ngrams(text, n=4):
        return set(text[i:i+n] for i in range(len(text)-n+1))
    
    # Calculate vocabulary richness
    df['unique_words'] = df['text_lower'].apply(lambda x: len(set(x.split())))
    df['total_words'] = df['text_lower'].apply(lambda x: len(x.split()))
    df['type_token_ratio'] = df['unique_words'] / df['total_words']
    
    # Calculate rare word frequency using TF-IDF
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(df['text_lower'])
    
    # Get high TF-IDF scores (potentially rare/unique phrases)
    max_tfidf_scores = np.array(tfidf_matrix.max(axis=1).todense()).flatten()
    df['max_tfidf'] = max_tfidf_scores
    
    # Mean TF-IDF (overall rarity)
    mean_tfidf_scores = np.asarray(tfidf_matrix.mean(axis=1)).flatten()
    df['mean_tfidf'] = mean_tfidf_scores
    
    return df

# Analyze uniqueness patterns
train_df_enhanced = analyze_text_uniqueness(train_df)

# Compare member vs non-member uniqueness
member_data = train_df_enhanced[train_df_enhanced['is_member'] == 1]
non_member_data = train_df_enhanced[train_df_enhanced['is_member'] == 0]

print("Text Uniqueness Analysis:")
print("=" * 50)
print(f"\nDuplicates:")
print(f"  Member texts with duplicates: {member_data['is_duplicate'].sum()}/{len(member_data)} ({member_data['is_duplicate'].mean()*100:.1f}%)")
print(f"  Non-member texts with duplicates: {non_member_data['is_duplicate'].sum()}/{len(non_member_data)} ({non_member_data['is_duplicate'].mean()*100:.1f}%)")

print(f"\nVocabulary Richness (Type-Token Ratio):")
print(f"  Member texts: {member_data['type_token_ratio'].mean():.3f} ± {member_data['type_token_ratio'].std():.3f}")
print(f"  Non-member texts: {non_member_data['type_token_ratio'].mean():.3f} ± {non_member_data['type_token_ratio'].std():.3f}")

print(f"\nRare Content (Max TF-IDF):")
print(f"  Member texts: {member_data['max_tfidf'].mean():.3f} ± {member_data['max_tfidf'].std():.3f}")
print(f"  Non-member texts: {non_member_data['max_tfidf'].mean():.3f} ± {non_member_data['max_tfidf'].std():.3f}")


Text Uniqueness Analysis:

Duplicates:
  Member texts with duplicates: 594/25000 (2.4%)
  Non-member texts with duplicates: 667/25000 (2.7%)

Vocabulary Richness (Type-Token Ratio):
  Member texts: 0.576 ± 0.071
  Non-member texts: 0.577 ± 0.071

Rare Content (Max TF-IDF):
  Member texts: 0.278 ± 0.073
  Non-member texts: 0.279 ± 0.074


In [8]:
#  Medical Domain-Specific Analysis
def analyze_medical_patterns(df):
    """Analyze medical terminology and discharge note structure"""
    
    df = df.copy()
    
    # Common medical terms/patterns (basic medical vocabulary)
    medical_terms = [
        'patient', 'diagnosis', 'treatment', 'medication', 'discharge', 'admission',
        'symptoms', 'condition', 'procedure', 'hospital', 'doctor', 'nurse',
        'blood', 'pressure', 'heart', 'lung', 'kidney', 'liver', 'brain',
        'surgery', 'therapy', 'pain', 'fever', 'infection', 'chronic', 'acute',
        'mg', 'ml', 'dose', 'daily', 'twice', 'prescribed', 'follow-up',
    ]
    
    # Structured sections common in discharge notes
    structured_patterns = [
        'discharge summary', 'chief complaint', 'history of present illness',
        'past medical history', 'medications', 'allergies', 'physical exam',
        'assessment', 'plan', 'disposition', 'follow up', 'instructions'
    ]
    
    # Calculate medical term density
    text_lower = df['text'].str.lower()
    df['medical_term_count'] = 0
    for term in medical_terms:
        df['medical_term_count'] += text_lower.str.count(term)
    
    df['medical_term_density'] = df['medical_term_count'] / df['text'].str.split().str.len()
    
    # Calculate structured section count
    df['structured_section_count'] = 0
    for pattern in structured_patterns:
        df['structured_section_count'] += text_lower.str.count(pattern)
    
    # Numeric pattern analysis (medical values, dates, etc.)
    df['numeric_count'] = text_lower.str.count(r'\\d+')
    df['date_pattern_count'] = text_lower.str.count(r'\\d{1,2}/\\d{1,2}/\\d{2,4}')
    df['time_pattern_count'] = text_lower.str.count(r'\\d{1,2}:\\d{2}')
    
    # Medication pattern count (basic patterns)
    df['medication_pattern_count'] = (
        text_lower.str.count(' mg') + 
        text_lower.str.count(' ml') + 
        text_lower.str.count(' mcg') +
        text_lower.str.count('daily') +
        text_lower.str.count('twice') +
        text_lower.str.count('q\\d+h')  # q8h, q12h patterns
    )
    
    # Capitalized words (could indicate medical acronyms/abbreviations)
    df['caps_word_count'] = df['text'].str.count(r'\\b[A-Z]{2,}\\b')
    
    return df

# Analyze medical patterns
train_df_medical = analyze_medical_patterns(train_df)

# Compare medical characteristics by class
member_medical = train_df_medical[train_df_medical['is_member'] == 1]
non_member_medical = train_df_medical[train_df_medical['is_member'] == 0]

print("Medical Domain Analysis:")
print("=" * 50)
print(f"\n  Medical Term Density:")
print(f"  Member texts: {member_medical['medical_term_density'].mean():.3f} ± {member_medical['medical_term_density'].std():.3f}")
print(f"  Non-member texts: {non_member_medical['medical_term_density'].mean():.3f} ± {non_member_medical['medical_term_density'].std():.3f}")

print(f"\n  Structured Sections:")
print(f"  Member texts: {member_medical['structured_section_count'].mean():.2f} ± {member_medical['structured_section_count'].std():.2f}")
print(f"  Non-member texts: {non_member_medical['structured_section_count'].mean():.2f} ± {non_member_medical['structured_section_count'].std():.2f}")

print(f"\n  Medical Abbreviations (CAPS):")
print(f"  Member texts: {member_medical['caps_word_count'].mean():.2f} ± {member_medical['caps_word_count'].std():.2f}")
print(f"  Non-member texts: {non_member_medical['caps_word_count'].mean():.2f} ± {non_member_medical['caps_word_count'].std():.2f}")

print(f"\n  Medication Patterns:")
print(f"  Member texts: {member_medical['medication_pattern_count'].mean():.2f} ± {member_medical['medication_pattern_count'].std():.2f}")
print(f"  Non-member texts: {non_member_medical['medication_pattern_count'].mean():.2f} ± {non_member_medical['medication_pattern_count'].std():.2f}")



Medical Domain Analysis:

  Medical Term Density:
  Member texts: 0.087 ± 0.035
  Non-member texts: 0.087 ± 0.035

  Structured Sections:
  Member texts: 2.57 ± 2.09
  Non-member texts: 2.57 ± 2.13

  Medical Abbreviations (CAPS):
  Member texts: 0.00 ± 0.00
  Non-member texts: 0.00 ± 0.00

  Medication Patterns:
  Member texts: 0.55 ± 1.47
  Non-member texts: 0.58 ± 1.52


In [9]:
# N-gram Rarity Analysis

def analyze_ngram_rarity(df, n_values=[2, 3, 4]):
    """Analyze n-gram frequency patterns that might indicate memorization"""
    
    df = df.copy()
    
    # Get all texts for computing global frequencies
    all_texts = df['text'].str.lower().tolist()
    
    for n in n_values:
        print(f"\nAnalyzing {n}-grams...")
        
        # Count all n-grams in the dataset
        all_ngrams = []
        for text in all_texts:
            words = text.split()
            ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
            all_ngrams.extend(ngrams)
        
        ngram_counts = Counter(all_ngrams)
        total_ngrams = len(all_ngrams)
        
        # For each text, calculate n-gram rarity scores
        rarity_scores = []
        rare_ngram_counts = []
        
        for text in df['text'].str.lower():
            words = text.split()
            if len(words) < n:
                rarity_scores.append(0)
                rare_ngram_counts.append(0)
                continue
                
            ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
            
            # Calculate average rarity (negative log frequency)
            text_rarity_score = 0
            rare_count = 0
            
            for ngram in ngrams:
                frequency = ngram_counts[ngram] / total_ngrams
                rarity = -math.log(frequency)
                text_rarity_score += rarity
                
                # Count rare n-grams (appear ≤ 2 times in entire dataset)
                if ngram_counts[ngram] <= 2:
                    rare_count += 1
            
            avg_rarity = text_rarity_score / len(ngrams) if ngrams else 0
            rarity_scores.append(avg_rarity)
            rare_ngram_counts.append(rare_count)
        
        df[f'{n}gram_avg_rarity'] = rarity_scores
        df[f'{n}gram_rare_count'] = rare_ngram_counts
        
        # Print top rare n-grams by class
        member_texts = df[df['is_member'] == 1]['text'].str.lower().tolist()
        non_member_texts = df[df['is_member'] == 0]['text'].str.lower().tolist()
        
        # Get n-grams from member vs non-member texts
        member_ngrams = []
        non_member_ngrams = []
        
        for text in member_texts:
            words = text.split()
            ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
            member_ngrams.extend(ngrams)
            
        for text in non_member_texts:
            words = text.split()
            ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
            non_member_ngrams.extend(ngrams)
        
        member_ngram_counts = Counter(member_ngrams)
        non_member_ngram_counts = Counter(non_member_ngrams)
        
        # Find n-grams that appear much more in one class than the other
        member_specific = []
        non_member_specific = []
        
        for ngram, count in member_ngram_counts.items():
            if count >= 3:  # Must appear at least 3 times
                other_count = non_member_ngram_counts.get(ngram, 0)
                if other_count == 0 or count / (other_count + 1) >= 3:  # 3x more frequent
                    member_specific.append((ngram, count, other_count))
        
        for ngram, count in non_member_ngram_counts.items():
            if count >= 3:
                other_count = member_ngram_counts.get(ngram, 0)
                if other_count == 0 or count / (other_count + 1) >= 3:
                    non_member_specific.append((ngram, count, other_count))
        
        print(f"  Class-specific {n}-grams:")
        print(f"    Member-specific: {len(member_specific)} unique {n}-grams")
        if member_specific:
            print(f"      Top examples: {sorted(member_specific, key=lambda x: x[1], reverse=True)[:3]}")
        
        print(f"    Non-member-specific: {len(non_member_specific)} unique {n}-grams")
        if non_member_specific:
            print(f"      Top examples: {sorted(non_member_specific, key=lambda x: x[1], reverse=True)[:3]}")
    
    return df

# Perform n-gram analysis
print("N-gram Rarity Analysis (Memorization Detection):")
print("=" * 60)

train_df_ngrams = analyze_ngram_rarity(train_df)

# Compare n-gram rarity by class
member_ngrams = train_df_ngrams[train_df_ngrams['is_member'] == 1]
non_member_ngrams = train_df_ngrams[train_df_ngrams['is_member'] == 0]

print(f"\nN-gram Rarity Comparison:")
for n in [2, 3, 4]:
    print(f"\n{n}-gram Analysis:")
    print(f"  Member avg rarity: {member_ngrams[f'{n}gram_avg_rarity'].mean():.3f} ± {member_ngrams[f'{n}gram_avg_rarity'].std():.3f}")
    print(f"  Non-member avg rarity: {non_member_ngrams[f'{n}gram_avg_rarity'].mean():.3f} ± {non_member_ngrams[f'{n}gram_avg_rarity'].std():.3f}")
    
    print(f"  Member rare {n}-grams per text: {member_ngrams[f'{n}gram_rare_count'].mean():.2f}")
    print(f"  Non-member rare {n}-grams per text: {non_member_ngrams[f'{n}gram_rare_count'].mean():.2f}")

# Visualize n-gram patterns
ngram_comparison_data = pd.concat([
    pd.DataFrame({
        'bigram_rarity': member_ngrams['2gram_avg_rarity'],
        'trigram_rarity': member_ngrams['3gram_avg_rarity'], 
        'fourgram_rarity': member_ngrams['4gram_avg_rarity'],
        'rare_bigrams': member_ngrams['2gram_rare_count'],
        'class': 'Member'
    }),
    pd.DataFrame({
        'bigram_rarity': non_member_ngrams['2gram_avg_rarity'],
        'trigram_rarity': non_member_ngrams['3gram_avg_rarity'],
        'fourgram_rarity': non_member_ngrams['4gram_avg_rarity'],
        'rare_bigrams': non_member_ngrams['2gram_rare_count'],
        'class': 'Non-Member'
    })
]).reset_index(drop=True)

# Create n-gram rarity visualization
bigram_chart = alt.Chart(ngram_comparison_data).mark_boxplot().encode(
    x=alt.X('class:N'),
    y=alt.Y('bigram_rarity:Q', title='Average Bigram Rarity'),
    color=alt.Color('class:N')
).properties(title='Bigram Rarity', width=200, height=180)

trigram_chart = alt.Chart(ngram_comparison_data).mark_boxplot().encode(
    x=alt.X('class:N'),
    y=alt.Y('trigram_rarity:Q', title='Average Trigram Rarity'),
    color=alt.Color('class:N')
).properties(title='Trigram Rarity', width=200, height=180)

(bigram_chart | trigram_chart).show()

N-gram Rarity Analysis (Memorization Detection):

Analyzing 2-grams...
  Class-specific 2-grams:
    Member-specific: 38249 unique 2-grams
      Top examples: [('right fallopian', 37, 11), ('lacrimal sac', 35, 9), ('resected using', 31, 8)]
    Non-member-specific: 39010 unique 2-grams
      Top examples: [('mr images', 51, 13), ('our pain', 38, 9), ('an international', 37, 10)]

Analyzing 3-grams...
  Class-specific 3-grams:
    Member-specific: 59378 unique 3-grams
      Top examples: [('hilar lymph node', 37, 10), ('brain. the patient', 36, 11), ('and renal failure.', 33, 10)]
    Non-member-specific: 61068 unique 3-grams
      Top examples: [('effect. the patient', 45, 14), ('during the 1-year', 34, 7), ('taken at the', 33, 6)]

Analyzing 4-grams...
  Class-specific 4-grams:
    Member-specific: 52466 unique 4-grams
      Top examples: [('the brain. the patient', 31, 6), ('showed no abnormalities, and', 30, 6), ('but the patient remained', 27, 8)]
    Non-member-specific: 54972 uni

alt.HConcatChart(...)